# Experiment 013: 7B SFT to GRPO, phase 1, seed 0 (Colab)

The first reportable 7b run. Frozen protocol v1, a100-80gb only, execution
settings identical to every other scale (pilot A verdict, 22 jul), oom ladder
off per the no-mutation rule. This run also pays for the shared 7b base evals
once, later 013 conditions reuse them automatically.

Measured expectation from the pilot: grpo training ~3.8h, whole phase ~10-12h,
so this probably spans two colab sessions. Everything is resume-safe:
reconnect, run all again, finished stages skip and training resumes from the
last checkpoint.

If the run OOMs despite the pilot margin: do NOT rerun as-is. Per the plan,
move the failed attempt to a failed_attempts folder and record it before starting
the next pre-registered config (4x2 + checkpointing) from step zero.

## 0. Compute units and mount Drive

Fill `cu_before` now. The open record is saved straight after Drive mounts,
so a disconnected session can be closed when the next runtime starts.

In [10]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 201.65   # from the resources panel
unit_rate = 6.77   # units per hour, also from the panel # e.g. 87.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '013_colab_7b_phase1', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260722_145839 | cu_before = 201.65
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved open session: 20260722_145839
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 7b pilot'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-eb66c540-3bb9-27c3-eddf-a34ac69e0a6f)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 1847, done.
remote: Counting objects: 100% (326/326), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 1847 (delta 190), reused 274 (delta 144), pack-reused 1521 (from 1)
Receiving objects: 100% (1847/1847), 56.98 MiB | 18.97 MiB/s, done.
Resolving deltas: 100% (1097/1097), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 160.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.8 MB/s eta 0:00:00
deps installed


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [5]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [6]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 60, 'need >=60GB free on drive for the 7b phases'

open record missing, rerun section 0 with cu_before filled
drive free: 179 GB


## 6. Build the data splits

In [7]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 22.2MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.57MB/s,  223kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.58MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 313kB/s, 40.7kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 313kB/s, 40.7kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 369675.35 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 288636.04 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 79.6MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 130MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B

## 7. Wiring check

In [8]:
!python scripts/run.py --config configs/013_colab_full_b7_seed0.yaml

env=COLAB mode=FULL scale=B7 condition=SFT_GRPO
model=Qwen/Qwen2.5-7B device=cuda stages=['base_eval', 'sft', 'sft_eval', 'grpo', 'grpo_eval']
steps=500 N=8 seed=0
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/013_COLAB_FULL_B7_sft_grpo_seed0/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/013_COLAB_FULL_B7_sft_grpo_seed0/
local smoke wiring ok


## 8. Full run

**If colab disconnects:** re-run this cell, finished stages skip.

In [11]:
# both cost fields must be filled before the long run starts
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/013_colab_full_b7_seed0.yaml --stage all
if _exit_code != 0:
    raise SystemExit(f'phase 1 failed with exit code {_exit_code}, see the OOM note in the header before retrying')

Streaming output truncated to the last 5000 lines.
│ │ letter of the │ agent of an   │               │              │           │ │
│ │ correct       │ end, which    │               │              │           │ │
│ │ option (A, B, │ effectively   │               │              │           │ │
│ │ C or D)       │ becomes the   │               │              │           │ │
│ │ inside        │ internal      │               │              │           │ │
│ │ <answer></an… │ final cause   │               │              │           │ │
│ │ tags.         │ of an agent's │               │              │           │ │
│ │               │ action. This  │               │              │           │ │
│ │ Aquinas       │ picture of a  │               │              │           │ │
│ │ argues that   │ cause within  │               │              │           │ │
│ │ every agent   │ an agent      │               │              │           │ │
│ │ acts for the  │ contrasts     │               │       

## 9. Peek at the results

In [12]:
import json, os
D = '/content/drive/MyDrive/inm363'
d = os.path.join(D, 'results', '013_COLAB_FULL_B7_sft_grpo_seed0')
sm = os.path.join(d, 'summary.json')
if os.path.exists(sm):
    j = json.load(open(sm))
    print('base', j['base_acc'])
    print('sft ', j['sft_acc'])
    print('grpo', j['grpo_acc'])
    print('probe base/sft/grpo:', j['base_logprob_acc'], j['sft_logprob_acc'], j['grpo_logprob_acc'])
else:
    print('no summary yet:', d)
be = os.path.join(D, 'results', 'base_evals', '013_COLAB_FULL_B7')
print('shared base evals:', os.listdir(be) if os.path.isdir(be) else 'not there yet')

base {'easy': 0.8691695108077361, 'medium': 0.735632183908046, 'hard': 0.5959595959595959}
sft  {'easy': 0.8816837315130831, 'medium': 0.7569786535303776, 'hard': 0.5883838383838383}
grpo {'easy': 0.8930602957906713, 'medium': 0.7766830870279147, 'hard': 0.6388888888888888}
probe base/sft/grpo: {'easy': 0.8976109215017065, 'medium': 0.812807881773399, 'hard': 0.5656565656565656} {'easy': 0.8998862343572241, 'medium': 0.8078817733990148, 'hard': 0.5681818181818182} {'easy': 0.9021615472127418, 'medium': 0.8078817733990148, 'hard': 0.5606060606060606}
shared base evals: ['base_preds.jsonl', 'base_eval.json', 'base_logprob_preds.jsonl', 'base_logprob.json']


## 10. Close the units record

In [14]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 168.69    # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

logged: {'session_id': '20260722_145839', 'ts_start': '2026-07-22 15:05', 'notebook': '013_colab_7b_phase1', 'colab_cu_before': 201.65, 'unit_rate': 6.77, 'ts_end': '2026-07-22 20:12', 'colab_cu_after': 168.69, 'colab_cu_used': 32.96, 'closed_on_resume': False}
